### Citation Dynamics
We further explor how citations differ **over time** between sharing and non-sharing articles; and whether different types of data sharing are associated with different citation dynamics.<br>
To do so, we calculate the number of citations in each calendar year post-publication for each article (ignoring "year 0" due to partial year effects). We then average these yearly citation counts across articles within each sharing category, and examine the differences.

In [ ]:
import sys
from pathlib import Path

# notebooks live in `analysis/`, which must be importable for `helpers` to resolve
_ANALYSIS_DIR = Path.cwd() if (Path.cwd() / "helpers").is_dir() else Path.cwd() / "analysis"
if str(_ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(_ANALYSIS_DIR))

from typing import Literal, Dict
from itertools import combinations
from copy import deepcopy

import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers import dataset
from helpers.config import (
    CLASS_COLORS, SHARING_CLASS_ORDER, BINARY_FEATURES,
    TITLE_FONT, AXIS_TITLE_FONT, AXIS_TICK_FONT, LEGEND_FONT, FONT_FAMILY,
    VENUE_IMPACT_METRIC, VENUE_IMAPCT_METRIC_NAME,
)
from helpers.plotting import save_figure

pio.renderers.default = "browser"

# flip to True to re-export this notebook's figures into `output/`
SAVE_FIGURES = False

combined, FEATURES_DF, CITATIONS_DF = dataset.load_or_build()

In [ ]:
def citations_per_year(
        articles: pd.DataFrame, grouping_col: str, cumulative: bool = True
) -> pd.DataFrame:
    """
    Extract citation counts per calendar-year post publication from OpenAlex metadata.
    Citations are stored in columns "year_X" where X is the number of years since publication, e.g. "year_0"
    for citations from the same year as article's publication.
    We ignore the article's publication year (year 0) and the current year (2026) to avoid
    partial-year effects.

    :param articles: pd.Datarme containing the grouping column and additional columns:
        - 'Citations20XX' columns with citation counts per year
        - 'PublicationYear' column with the publication year of each article
    :param grouping_col: str, name of the column to group by (e.g., "is_sharing_data" or "data_sharing_class")
    :param cumulative: bool, whether to return cumulative citation counts (default: True)
    :return: pd.DataFrame with columns "year_X" and the same index as the input articles DataFrame
    """
    cit_cols = [c for c in articles.columns if c.startswith("Citations20") and not c.endswith("2026")]
    articles_long = articles.reset_index().melt(
        id_vars=['index', grouping_col, 'PublicationYear'],
        value_vars=cit_cols,
        var_name='CitationYear_Str',
        value_name='Count'
    )
    articles_long['CitationYear'] = articles_long['CitationYear_Str'].str.extract(r'(\d{4})').astype(int)
    articles_long['RelYear'] = articles_long['CitationYear'] - articles_long['PublicationYear']
    articles_long = articles_long.loc[articles_long['RelYear'] > 0]    # drop citations from before publication year
    cit_dynamics = articles_long.pivot_table(
        index=['index', grouping_col],
        columns='RelYear',
        values='Count',
        aggfunc='sum'
    )
    cit_dynamics.columns = [int(c) for c in cit_dynamics.columns]
    if cumulative:
        cit_dynamics = cit_dynamics.cumsum(axis=1)
    return cit_dynamics

#### (1) Sharing vs. Non-Sharing Articles
We compare the cumulative citation counts per year between articles that share data vs. those that do not.

In [ ]:
GROUPING_COL = "is_sharing_data"

# cumulative shows the total citations up to (and including) each year (stock)
# non-cumulative shows the "velocity" of citations per year (flow)
cit_dynamics_sharing = citations_per_year(
    combined, grouping_col=GROUPING_COL,
    cumulative=True
)
# drop years with fewer than 20 articles combined
cit_dynamics_sharing = cit_dynamics_sharing.loc[:, cit_dynamics_sharing.count(axis=0) >= 20]

year_stats = dict()
for year in cit_dynamics_sharing.columns:
    non_sharing_vals = cit_dynamics_sharing.xs(False, level=GROUPING_COL)[year].dropna()
    non_sharing_is_normal = stats.shapiro(non_sharing_vals).pvalue > 0.05
    sharing_vals = cit_dynamics_sharing.xs(True, level=GROUPING_COL)[year].dropna()
    sharing_is_normal = stats.shapiro(sharing_vals).pvalue > 0.05
    n1, n2 = len(non_sharing_vals), len(sharing_vals)
    yearly_ttest = pg.ttest(non_sharing_vals, sharing_vals, paired=False, alternative="less", correction=True)
    yearly_mwtest = pg.mwu(non_sharing_vals, sharing_vals, alternative="less")
    U_nonshare = yearly_mwtest.loc["MWU", "U_val"]
    U_share = n1 * n2 - U_nonshare
    rank_biserial_corr = (U_share - U_nonshare) / (n1 * n2)
    cles = U_share / (n1 * n2)
    year_stats[year] = {
        "non_sharing_count": n1,
        "non_sharing_is_normal": non_sharing_is_normal,
        "non_sharing_median": non_sharing_vals.median(),
        "non_sharing_mean": non_sharing_vals.mean(),
        "non_sharing_sem": non_sharing_vals.sem(),

        "sharing_count": n2,
        "sharing_is_normal": sharing_is_normal,
        "sharing_median": sharing_vals.median(),
        "sharing_mean": sharing_vals.mean(),
        "sharing_sem": sharing_vals.sem(),

        "t_stat": yearly_ttest.loc["T_test", "T"],
        "t_dof": yearly_ttest.loc["T_test", "dof"],
        "t_p_raw": yearly_ttest.loc["T_test", "p_val"],
        "t_cohen_d": yearly_ttest.loc["T_test", "cohen_d"],

        "U_nonshare": U_nonshare,
        "U_share": U_share,
        "mw_p_raw": yearly_mwtest.loc["MWU", "p_val"],
        "r_rb": abs(yearly_mwtest.loc["MWU", "RBC"]),
        "cles": yearly_mwtest.loc["MWU", "CLES"],
    }

year_stats = pd.DataFrame.from_dict(year_stats, orient='index').reset_index().rename(columns={"index": "year"})
t_reject, t_p_corrected, _, _ = multipletests(year_stats['t_p_raw'], alpha=0.05, method='fdr_bh')
mw_reject, mw_p_corrected, _, _ = multipletests(year_stats['mw_p_raw'], alpha=0.05, method='fdr_bh')
year_stats['t_p_corrected'] = t_p_corrected
year_stats['t_is_significant'] = t_reject
year_stats['mw_p_corrected'] = mw_p_corrected
year_stats['mw_is_significant'] = mw_reject
column_order = [
    "year",
    "non_sharing_is_normal", "sharing_is_normal",
    "t_is_significant", "t_p_corrected",
    "mw_is_significant", "mw_p_corrected",
    "t_stat", "t_dof", "t_p_raw", "t_cohen_d",
    "U_share", "U_nonshare", "r_rb", "cles",
    "non_sharing_count", "non_sharing_mean", "non_sharing_sem", "non_sharing_median",
    "sharing_count", "sharing_mean", "sharing_sem", "sharing_median",
]
year_stats = year_stats[column_order]

year_stats

In [ ]:
LINE_ANNOTATION_X = 3

cit_dynamics_sharing_fig = go.Figure()
for is_share in cit_dynamics_sharing.index.get_level_values(GROUPING_COL).unique().sort_values():
    subset = cit_dynamics_sharing.xs(is_share, level=GROUPING_COL)
    count_per_year = subset.count(axis=0)
    mean_per_year = subset.mean(axis=0)
    sem_per_year = subset.sem(axis=0)
    mean_citations_per_year = subset.mean(axis=0)
    label = "Sharing" if is_share else "Not Sharing"
    color = CLASS_COLORS[label.upper()]
    cit_dynamics_sharing_fig.add_trace(go.Scatter(
        name=label, legendgroup=label,
        x=mean_per_year.index,
        y=mean_per_year.values,
        error_y=dict(type='data', array=sem_per_year.values, width=5, visible=True,),
        marker=dict(
            color=color,
            # size=np.sqrt(count_per_year.values),
        ),
        mode='lines+markers',
    ))
    # annotate the lines
    if is_share:
        ann_y = mean_per_year[LINE_ANNOTATION_X] + 1.5 * sem_per_year[LINE_ANNOTATION_X]
        xanchor = "right"
        yanchor = "bottom"
    else:
        ann_y = mean_per_year[LINE_ANNOTATION_X] - 1.5 * sem_per_year[LINE_ANNOTATION_X]
        xanchor = "left"
        yanchor = "top"
    cit_dynamics_sharing_fig.add_annotation(
        x=LINE_ANNOTATION_X, xanchor=xanchor, xref="x",
        y=ann_y, yanchor=yanchor, yref="y",
        text=label, font={**AXIS_TITLE_FONT, "color": color},
        showarrow=False,
    )

# add significance asterisks
SIGNIFICANCE_ANNOTATION_Y = 40
for _, row in year_stats.iterrows():
    if row["mw_p_corrected"] >= 0.1:
        continue
    fontsize = 22
    yanchor = "middle"
    if row["mw_p_corrected"] < 0.001:
        ann_text = "<b>***</b>"
    elif row["mw_p_corrected"] < 0.01:
        ann_text = "<b>**</b>"
    elif row["mw_p_corrected"] < 0.05:
        ann_text = "<b>*</b>"
    else:
        ann_text = "\u2020"     # dagger for p < 0.
        fontsize = 16
        yanchor = "bottom"
    cit_dynamics_sharing_fig.add_annotation(
        x=row["year"], xanchor="center", xref="x",
        y=SIGNIFICANCE_ANNOTATION_Y, yanchor=yanchor, yref="y",
        text=ann_text, font={**TITLE_FONT, "size": fontsize},
        showarrow=False,
    )

cit_dynamics_sharing_fig.update_layout(
    width=800, height=400,
    title=dict(
        text="Citation Dynamics: Sharing vs. Non-Sharing Articles",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top"
    ),
    xaxis=dict(
        title=dict(text="Years Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Mean Cumulative Citation per Year", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        showgrid=True, gridcolor='lightgrey', gridwidth=1.5,
        zeroline=False,
    ),
    legend=dict(
        title=dict(text="Data Sharing", font=AXIS_TITLE_FONT),
        font=LEGEND_FONT,
        x=1.0, xanchor="right", y=0.4, yanchor="top",
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black', borderwidth=1,
        visible=False,
    ),
    margin=dict(t=75, b=50, l=50, r=25, pad=0),
    template="plotly_white"
)
cit_dynamics_sharing_fig.show()

In [ ]:
if SAVE_FIGURES:
    save_figure(cit_dynamics_sharing_fig, "citation_dynamics_sharing_vs_nonsharing.png", width=800, height=400)

##### Source of the Difference
So we found a difference in yearly cumulative citations between sharing and non-sharing articles.<br>
But where does this difference come from? Is it because the sharing articles get cited more frequently in each year (i.e., higher "velocity" of citations)? Or is it because they got an initian citation boost which then accumulates over time (i.e., higher "stock" of citations)?<br>
To answer this question, we repeat the above analysis but now looking at **non-cumulative** citation counts per year (i.e., the "velocity" of citations).

In [ ]:
GROUPING_COL = "is_sharing_data"

# cumulative shows the total citations up to (and including) each year (stock)
# non-cumulative shows the "velocity" of citations per year (flow)
cit_dynamics_sharing = citations_per_year(
    combined, grouping_col=GROUPING_COL,
    cumulative=False
)

year_stats = dict()
for year in cit_dynamics_sharing.columns:
    non_sharing_vals = cit_dynamics_sharing.xs(False, level=GROUPING_COL)[year].dropna()
    sharing_vals = cit_dynamics_sharing.xs(True, level=GROUPING_COL)[year].dropna()
    n1, n2 = len(non_sharing_vals), len(sharing_vals)
    year_pair_test = stats.mannwhitneyu(non_sharing_vals, sharing_vals, alternative="less")
    U_nonshare = year_pair_test.statistic
    U_share = n1 * n2 - U_nonshare
    rank_biserial_corr = (U_share - U_nonshare) / (n1 * n2)
    cles = U_share / (n1 * n2)
    year_stats[year] = {
        "non_sharing_count": n1,
        "non_sharing_mean": non_sharing_vals.mean(),
        "non_sharing_sem": non_sharing_vals.sem(),
        "sharing_count": n2,
        "sharing_mean": sharing_vals.mean(),
        "sharing_sem": sharing_vals.sem(),
        "U_nonshare": U_nonshare,
        "p_raw": year_pair_test.pvalue,
        "r_rb": rank_biserial_corr,
        "cles": cles,
    }

year_stats = pd.DataFrame.from_dict(year_stats, orient='index').reset_index().rename(columns={"index": "year"})
reject, p_corrected, _, _ = multipletests(year_stats['p_raw'], alpha=0.05, method='fdr_bh')
year_stats['p_corrected'] = p_corrected
year_stats['is_significant'] = reject
column_order = [
    "year",
    "is_significant", "p_corrected", "U_nonshare", "r_rb", "cles",
    "non_sharing_count", "non_sharing_mean", "non_sharing_sem",
    "sharing_count", "sharing_mean", "sharing_sem",
    "p_raw"
]
year_stats = year_stats[column_order]

year_stats

### (2) Different Types of Data Sharing

In [ ]:
GROUPING_COL = "data_sharing_class"

cit_dynamics_sharing_class = citations_per_year(
    combined.loc[combined["is_sharing_data"]], grouping_col=GROUPING_COL
)

year_stats = dict()
for year in cit_dynamics_sharing_class.columns:
    year_stats[year] = dict()
    class_values = []
    for share_class in cit_dynamics_sharing_class.index.get_level_values(GROUPING_COL).unique():
        subset = (cit_dynamics_sharing_class.xs(share_class, level=GROUPING_COL)[year]).dropna()
        if subset.empty:
            continue
        year_stats[year][f"{share_class}_count"] = subset.count()
        year_stats[year][f"{share_class}_mean"] = subset.mean()
        year_stats[year][f"{share_class}_sem"] = subset.sem()
        class_values.append(subset)
    sharegroup_test = stats.kruskal(*class_values)
    year_stats[year]["W"] = sharegroup_test.statistic
    year_stats[year]["p_raw"] = sharegroup_test.pvalue

year_stats = pd.DataFrame.from_dict(year_stats, orient='index')
reject, p_corrected, _, _ = multipletests(year_stats['p_raw'], alpha=0.05, method='fdr_bh')
year_stats['p_corrected'] = p_corrected
year_stats['is_significant'] = reject
column_order = [
    "is_significant", "p_corrected", "W", "p_raw",
    *[f"{share_class}_{stat}" for share_class in ["FIXATION", "TRIAL", "PARTICIPANT"] for stat in ["count", "mean", "sem"]]
]
year_stats = year_stats[column_order]
year_stats

In [ ]:
cit_dynamics_sharing_class_fig = go.Figure()
for share_class in cit_dynamics_sharing_class.index.get_level_values(GROUPING_COL).unique():
    subset = cit_dynamics_sharing_class.xs(share_class, level=GROUPING_COL)
    count_per_year = subset.count(axis=0)
    mean_per_year = subset.mean(axis=0)
    sem_per_year = subset.sem(axis=0)
    mean_citations_per_year = subset.mean(axis=0)
    cit_dynamics_sharing_class_fig.add_trace(go.Scatter(
        name=share_class, legendgroup=share_class,
        x=mean_per_year.index,
        y=mean_per_year.values,
        error_y=dict(type='data', array=sem_per_year.values, width=5, visible=True,),
        marker=dict(
            color=CLASS_COLORS[share_class],
            size=count_per_year.values
        ),
        mode='lines+markers',
    ))

cit_dynamics_sharing_class_fig.update_layout(
    width=800, height=400,
    title=dict(
        text="Citation Dynamics: Different Types of Data Sharing",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top"
    ),
    xaxis=dict(
        title=dict(text="Years Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Mean Cumulative Citation per Year", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    legend=dict(
        title=dict(text="Data Sharing", font=AXIS_TITLE_FONT),
        font=LEGEND_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top",
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black', borderwidth=1,
    ),
    margin=dict(t=75, b=50, l=50, r=25, pad=0),
    template="plotly_white"
)
cit_dynamics_sharing_class_fig.show()

In [ ]:
if SAVE_FIGURES:
    save_figure(cit_dynamics_sharing_class_fig, "citation_dynamics_sharing_class.png", width=800, height=400)